# 07 — Challengers

In [ ]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [ ]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

In [ ]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

Whether a constrained gradient-boosting challenger beats the card out of time by enough to matter, what it finds that the card does not (interactions, finer thresholds), and whether that signal can be translated back into attributes the card can carry — the ML-to-compliant-attributes method. The challenger descends the same Bernoulli log-loss; it differs in function class and search.

Decisions: **N — constraints and hyper-parameters**, **O — which GBM thresholds become card attributes**, **P — the champion decision memo**.

In [ ]:
frame = load_frame()
tr, va, oot = (frame[frame.split == s] for s in ("train", "validate", "oot"))
card, specs = S.load_card(MODELS / "champion"); card_attrs = card.attributes
cands = json.load(open(MODELS / "candidate_attributes.json"))["candidates"]
sc_oot = S.score(oot, card, specs); gini_card = V.delong_ci(oot["bad"], sc_oot["score"])
print("card OOT gini %.4f [%.4f, %.4f] on %d attributes" % (gini_card["gini"], *gini_card["gini_ci"], len(card_attrs)))

### Decision N — Constraints and hyper-parameters

Monotone constraints from the registry are on by default (required for reason codes). Depth, minimum child weight and learning rate are the capacity dials; early stopping is on the validation vintage, never on OOT. Consider interaction constraints that keep attributes of one family together.

**Your reasoning:**

_(write here)_

In [ ]:
XGB_PARAMS = {"max_depth": 4, "min_child_weight": 50, "eta": 0.05, "subsample": 0.8, "colsample_bytree": 0.8}
xgb_all = C.fit_xgb(tr, tr["bad"], va, va["bad"], cands, monotone=True, params=XGB_PARAMS)
xgb_card = C.fit_xgb(tr, tr["bad"], va, va["bad"], card_attrs, monotone=True, params=XGB_PARAMS)
lgb_all = C.fit_lgbm(tr, tr["bad"], va, va["bad"], cands, monotone=True)
rows = []
for name, model, pred in (("xgb, all candidates", xgb_all, C.predict_xgb), ("xgb, card attributes only", xgb_card, C.predict_xgb), ("lgbm, all candidates", lgb_all, C.predict_lgbm)):
    p = pred(model, oot); m = V.delong_ci(oot["bad"], p, higher_is_safer=False)
    rows.append({"model": name, "best_iteration": int(model.best_iteration), "gini_oot": round(m["gini"], 4), "gini_ci": tuple(round(x, 4) for x in m["gini_ci"]),
                 "lift_vs_card": round(m["gini"] - gini_card["gini"], 4)})
rows.append({"model": "champion card", "best_iteration": None, "gini_oot": round(gini_card["gini"], 4), "gini_ci": tuple(round(x, 4) for x in gini_card["gini_ci"]), "lift_vs_card": 0.0})
display(pd.DataFrame(rows))
p_xgb = C.predict_xgb(xgb_all, oot); display(pd.Series(V.delong_test(oot["bad"], -p_xgb, sc_oot["score"])))

In [ ]:
# Evidence — what the GBM uses, and whether its reasons agree with the card's
sv, bias = C.shap_values(xgb_all, oot, max_rows=20000)
imp = C.shap_importance(sv); display(imp.head(20))
imp.to_csv(config.DOCS_DIR / "exhibits" / "shap_importance_xgb.csv", index=False)
sample = oot.loc[sv.index[:2000]]
rc_card = S.reason_codes(sample, card, specs, meanings=MEANINGS); rc_gbm = C.gbm_reason_codes(sv.loc[sample.index], meanings=MEANINGS)
agree = np.mean([len({rc_card.iloc[i, j].split(" [")[0] for j in range(4)} & set(rc_gbm.iloc[i, :4])) for i in range(len(sample))])
print(f"mean overlap of top-4 reason codes, card vs GBM: {agree:.2f} of 4")

In [ ]:
# Evidence — how much of the GBM is interaction (what an additive card cannot carry), and a surrogate tree
inter = C.interaction_strength(xgb_all, cands, oot, max_rows=3000); display(inter.head(15))
tree, fidelity = C.surrogate_tree(oot, p_xgb, cands, max_depth=3)
print(f"surrogate tree fidelity R2 = {fidelity:.3f}"); print(C.surrogate_rules(tree, cands))

### Decision O — Thresholds that become attributes

The GBM's highest-gain splits are candidate bin edges or flags the card did not have. Choose which to carry back: re-bin that attribute with the threshold as a forced edge (NB04 `OVERRIDES`), or add the flag as a new attribute, then refit the card and measure the Gini recovered.

**Your reasoning:**

_(write here)_

In [ ]:
cand_flags = C.splits_to_attributes(xgb_all, tr, top_k=10); display(cand_flags)
KEEP_FLAGS = cand_flags.candidate.tolist()[:5]         # edit
tr2, va2, oot2 = (C.add_split_flags(d, cand_flags[cand_flags.candidate.isin(KEEP_FLAGS)]) for d in (tr, va, oot))
specs2 = dict(specs)
for f in KEEP_FLAGS:
    specs2[f] = B.fit_numeric_with_edges(tr2[f], tr2["bad"], [0.5], name=f)
W2 = B.transform_woe(tr2, specs2)
sel2, log2 = S.stepwise_select(W2, tr2["bad"], card_attrs + KEEP_FLAGS, p_enter=0.01, max_corr=0.7, max_attributes=20)
card2 = S.fit_logistic(W2, tr2["bad"], sel2)
g2 = V.delong_ci(oot2["bad"], S.score(oot2, card2, specs2)["score"])
print("card + GBM flags: OOT gini %.4f [%.4f, %.4f] with %d attributes (was %.4f)" % (g2["gini"], *g2["gini_ci"], len(sel2), gini_card["gini"]))
display(S.coefficient_table(card2))

### Decision P — Champion decision memo

Write the memo here: the lift with its interval at the out-of-time level, what the GBM found, what the card recovered after translation, the explainability and stability cost of each option, and the decision (keep the card, keep the card with translated attributes, or promote the GBM with SHAP reason codes).

**Your memo:**

_(write here)_

In [ ]:
xgb_all.save_model(str(MODELS / "challenger_xgb.json"))
json.dump({"xgb_params": XGB_PARAMS, "candidates": cands, "kept_flags": KEEP_FLAGS, "gini_card_oot": gini_card["gini"],
           "gini_card_plus_flags_oot": g2["gini"]}, open(MODELS / "challenger_decisions.json", "w"), indent=1)
print("saved", MODELS / "challenger_xgb.json")